# 📈 Tracking Progress — Weeks and Months at a Glance

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gerolfziegenhain/climbing-science/main?labpath=notebooks%2F06_progress_tracker.ipynb)

## Introduction

Training without tracking is guessing. Finger strength adapts slowly —
typical gains are **1–2 % per month** for intermediate climbers — so
day-to-day noise easily masks real progress. A longitudinal view cuts
through that noise.

**What to measure:**

| Metric | Why |
|---|---|
| **MVC-7** (7-second max voluntary contraction) | Gold standard for finger strength |
| **Bodyweight** | Needed for power-to-weight ratio |
| **Power-to-Weight ratio** (MVC-7 / BW) | Predicts climbing grade better than raw force |

This notebook loads your measurements — either from a directory of
Tindeq Progressor JSON files or from a simple manual table — and
produces trend plots, grade predictions, and early-warning checks for
stagnation or overtraining.


## Your Data

In [ ]:
# Option A: Directory of Tindeq JSON files
SESSION_DIR = None  # e.g. "/path/to/sessions/"

# Option B: Manual tracking data (if no device)
# Enter your MVC-7 measurements over time
MANUAL_DATA = [
    {"date": "2026-01-15", "mvc7_kg": 95, "bodyweight_kg": 72},
    {"date": "2026-02-01", "mvc7_kg": 98, "bodyweight_kg": 71.5},
    {"date": "2026-02-15", "mvc7_kg": 101, "bodyweight_kg": 71},
    {"date": "2026-03-01", "mvc7_kg": 103, "bodyweight_kg": 71},
    {"date": "2026-03-15", "mvc7_kg": 105, "bodyweight_kg": 71},
    {"date": "2026-04-01", "mvc7_kg": 108, "bodyweight_kg": 70.5},
]


## Load Data

In [ ]:
%matplotlib inline

import datetime as dt

try:
    import pandas as pd
except ImportError:
    raise SystemExit("pandas is required — pip install pandas")

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

from climbing_science.strength import mvc7_to_grade, power_to_weight
from climbing_science.adapters.tindeq import load_all, extract_mvc7


def _build_timeline_from_tindeq(directory: str) -> pd.DataFrame:
    """Load all Tindeq sessions and extract MVC-7 + metadata."""
    sessions = load_all(directory)
    rows = []
    for sess in sessions:
        mvc7 = extract_mvc7(sess)
        meta = getattr(sess, "metadata", {}) or {}
        date_str = meta.get("date") or meta.get("timestamp", "")
        bw = meta.get("bodyweight_kg", None)
        if date_str:
            date = pd.to_datetime(date_str).normalize()
        else:
            date = pd.NaT
        rows.append({"date": date, "mvc7_kg": mvc7, "bodyweight_kg": bw})
    df = pd.DataFrame(rows).dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df


def _build_timeline_from_manual(data: list) -> pd.DataFrame:
    """Convert manual tracking list to DataFrame."""
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)


# Build the timeline
if SESSION_DIR:
    timeline = _build_timeline_from_tindeq(SESSION_DIR)
    print(f"✅ Loaded {len(timeline)} sessions from {SESSION_DIR}")
else:
    timeline = _build_timeline_from_manual(MANUAL_DATA)
    print(f"✅ Using manual data — {len(timeline)} data points")

timeline["pw_ratio"] = timeline.apply(
    lambda r: power_to_weight(r["mvc7_kg"], r["bodyweight_kg"])
    if pd.notna(r.get("bodyweight_kg")) else None,
    axis=1,
)

timeline


## MVC-7 Progression

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

dates_num = mdates.date2num(timeline["date"])
mvc7 = timeline["mvc7_kg"].values

ax.plot(timeline["date"], mvc7, "o-", color="#2563eb", linewidth=2, markersize=8, label="MVC-7")

# Linear trend
coeffs = np.polyfit(dates_num, mvc7, 1)
trend_line = np.polyval(coeffs, dates_num)
ax.plot(timeline["date"], trend_line, "--", color="#f59e0b", linewidth=1.5, label="Linear trend")

ax.set_title("MVC-7 Over Time", fontsize=14, fontweight="bold")
ax.set_ylabel("MVC-7 (kg)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
fig.autofmt_xdate()
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary stats
gain_abs = mvc7[-1] - mvc7[0]
gain_pct = (gain_abs / mvc7[0]) * 100
days_span = (timeline["date"].iloc[-1] - timeline["date"].iloc[0]).days
print(f"Absolute gain:  {gain_abs:+.1f} kg")
print(f"Relative gain:  {gain_pct:+.1f} %")
print(f"Over {days_span} days ({days_span / 7:.1f} weeks)")
print(f"Rate:           {gain_abs / max(days_span / 7, 1):.2f} kg / week")


## Power-to-Weight Progression

In [ ]:
has_pw = timeline["pw_ratio"].notna()

if has_pw.any():
    pw_df = timeline[has_pw].copy()

    fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.plot(pw_df["date"], pw_df["pw_ratio"], "s-", color="#16a34a",
             linewidth=2, markersize=8, label="P:W ratio")
    ax1.set_ylabel("Power-to-Weight ratio")
    ax1.set_title("Power-to-Weight Over Time", fontsize=14, fontweight="bold")

    # Grade prediction as secondary annotation
    for _, row in pw_df.iterrows():
        grade = mvc7_to_grade(row["mvc7_kg"])
        ax1.annotate(grade, (row["date"], row["pw_ratio"]),
                     textcoords="offset points", xytext=(0, 12),
                     fontsize=8, ha="center", color="#6b7280")

    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    fig.autofmt_xdate()
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No bodyweight data available — cannot compute P:W ratio.")


## Trend Analysis & Projection

In [ ]:
slope_per_day = coeffs[0]  # kg per day from the linear fit
slope_per_week = slope_per_day * 7

print(f"Linear trend: {slope_per_week:+.2f} kg / week  ({slope_per_day * 30:.2f} kg / month)")
print()

# Project forward to common targets
targets = [110, 120, 130, 140, 150]
current = mvc7[-1]
last_date = timeline["date"].iloc[-1]

print("Projected milestones (at current rate):")
print("-" * 45)
for target in targets:
    if target <= current:
        print(f"  {target} kg  — already reached ✅")
    elif slope_per_day <= 0:
        print(f"  {target} kg  — trend is flat/negative ⚠️")
    else:
        days_needed = (target - current) / slope_per_day
        target_date = last_date + pd.Timedelta(days=days_needed)
        weeks = days_needed / 7
        print(f"  {target} kg  — ~{weeks:.0f} weeks  (≈ {target_date.strftime('%B %Y')})")


## Grade Prediction Timeline

In [ ]:
timeline["predicted_grade"] = timeline["mvc7_kg"].apply(mvc7_to_grade)

fig, ax = plt.subplots(figsize=(10, 5))

grades = timeline["predicted_grade"].tolist()
unique_grades = sorted(set(grades), key=lambda g: grades.index(g))
grade_to_num = {g: i for i, g in enumerate(unique_grades)}
grade_nums = [grade_to_num[g] for g in grades]

ax.step(timeline["date"], grade_nums, where="post", color="#7c3aed",
        linewidth=2, marker="D", markersize=8)
ax.set_yticks(range(len(unique_grades)))
ax.set_yticklabels(unique_grades)
ax.set_title("Predicted Grade Over Time", fontsize=14, fontweight="bold")
ax.set_ylabel("Grade (Font / V-Scale)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
fig.autofmt_xdate()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for _, row in timeline.iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')}  MVC-7 = {row['mvc7_kg']:.1f} kg  →  {row['predicted_grade']}")


## Monthly Summary

In [ ]:
timeline["month"] = timeline["date"].dt.to_period("M")

monthly = timeline.groupby("month").agg(
    n=("mvc7_kg", "count"),
    avg_mvc7=("mvc7_kg", "mean"),
    best_mvc7=("mvc7_kg", "max"),
    worst_mvc7=("mvc7_kg", "min"),
    avg_bw=("bodyweight_kg", "mean"),
    avg_pw=("pw_ratio", "mean"),
).round(1)

monthly.index = monthly.index.astype(str)
print(monthly.to_string())


## Warnings & Red Flags

In [ ]:
warnings_found = []

# 1. Stagnation: no improvement over 4+ weeks
if len(timeline) >= 2:
    last_val = timeline["mvc7_kg"].iloc[-1]
    for i in range(len(timeline) - 2, -1, -1):
        if timeline["mvc7_kg"].iloc[i] < last_val - 0.5:
            break
        days_flat = (timeline["date"].iloc[-1] - timeline["date"].iloc[i]).days
        if days_flat >= 28:
            warnings_found.append(
                f"⚠️  STAGNATION: MVC-7 has not improved in {days_flat} days "
                f"(last {timeline['mvc7_kg'].iloc[i]:.1f} → now {last_val:.1f} kg)."
            )
            break

# 2. Large drops (> 5 % between consecutive measurements)
for i in range(1, len(timeline)):
    prev = timeline["mvc7_kg"].iloc[i - 1]
    curr = timeline["mvc7_kg"].iloc[i]
    drop_pct = (prev - curr) / prev * 100
    if drop_pct > 5:
        d = timeline["date"].iloc[i].strftime("%Y-%m-%d")
        warnings_found.append(
            f"🔴 LARGE DROP on {d}: {prev:.1f} → {curr:.1f} kg ({drop_pct:.1f} % loss). "
            "Check for illness, injury, or measurement error."
        )

# 3. Overtraining signal: bodyweight AND force both dropping
if timeline["bodyweight_kg"].notna().sum() >= 2:
    bw = timeline.dropna(subset=["bodyweight_kg"])
    if len(bw) >= 2:
        bw_trend = bw["bodyweight_kg"].iloc[-1] - bw["bodyweight_kg"].iloc[0]
        mvc_trend = bw["mvc7_kg"].iloc[-1] - bw["mvc7_kg"].iloc[0]
        if bw_trend < -1 and mvc_trend < 0:
            warnings_found.append(
                f"🟡 OVERTRAINING SIGNAL: Bodyweight dropped {bw_trend:+.1f} kg "
                f"AND MVC-7 dropped {mvc_trend:+.1f} kg. "
                "Consider a deload week and nutrition check."
            )

if warnings_found:
    print("\n".join(warnings_found))
else:
    print("✅ No red flags detected. Progress looks healthy!")


## Key Takeaways

**What good progress looks like:**

- **1–2 % MVC-7 gain per month** is solid for intermediate climbers.
- **3–5 %** is excellent and typical during the first 6 months of
  structured hangboard training.
- **> 5 %** per month is beginner gains — enjoy them while they last.

**When to deload:**

- After **3–4 weeks** of progressive overload, schedule 1 easy week.
- If MVC-7 stagnates for **4+ weeks**, a deload often restarts progress.
- If bodyweight and force **both drop**, rest more and eat more.

**Common pitfalls:**

- Testing too often (max once per week for reliable data).
- Ignoring bodyweight changes — a 2 kg cut can mask real strength gains.
- Comparing raw force instead of power-to-weight ratio.


## References

- López-Rivera, E. & González-Badillo, J.J. (2012). *The effects of two
  maximum grip strength training methods using the same effort duration
  and rest time on grip endurance in elite climbers.* Sports Technology,
  5(3-4), 100–110.
- Lattice Training (2023). *Finger Strength Benchmarking Protocol.*
  latticetraining.com
- Levernier, G. & Laffaye, G. (2019). *Four weeks of finger grip
  strength training in rock climbers.* Journal of Strength and
  Conditioning Research, 33(7), 1863–1871.
- strengthclimbing.com — MVC-7 testing and grade prediction models.
